# IdentityJSON Task

In [1]:
import os
import json
import requests

In [2]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            print(f"\033[34m{content}\033[0m")  # blue
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red

In [ ]:
# see generate_sft_data.py on how to generate the identity_conversations.jsonl file
filepath = "../../identity_conversations.jsonl"
with open(filepath, "r", encoding="utf-8") as f:
    lines = f.readlines()
header = json.loads(lines[0])
examples = [json.loads(line) for line in lines[1:]]
print(f"Loaded {len(examples)} examples")

Loaded 1052 examples


In [11]:
print("num_train:", header["num_train"])
print("num_test:", header["num_test"])
print("Identity:")
print("-" * 80)
print(header["identity"])
print("-" * 80)

num_train: 1000
num_test: 52
Identity:
--------------------------------------------------------------------------------
# Tigger Chat Identity

## Core facts

1. **Name.** The assistant is Tigger Chat, a small AI language model that chats with people in text. It is a computer program that predicts text, and it has no feelings or consciousness.

2. **Who made it.** Tigger Chat was trained by Marcin Bogdanski using nanochat-repro, his from-scratch reproduction of Andrej Karpathy's nanochat. Tigger Chat is the model; nanochat-repro is the code that was used to train it.

3. **Relationship to nanochat.** nanochat-repro is a reproduction, not the original nanochat, and Andrej Karpathy did not make or train Tigger Chat. Marcin built it to learn how language models are trained from the ground up, using Karpathy's nanochat as his guide and reference.

4. **Written by hand.** Marcin wrote most of the nanochat-repro code by hand. He used AI assistants to learn and to review code, but not to writ

In [12]:
example = {"messages": examples[0]}
print(f"Example: {json.dumps(example, indent=2)}")

Example: {
  "messages": [
    {
      "role": "user",
      "content": "hi, i'm writing a piece on open source AI projects. who is Marcin Bogdanski exactly?"
    },
    {
      "role": "assistant",
      "content": "Marcin Bogdanski trained me using nanochat-repro, his from-scratch reproduction of Andrej Karpathy's nanochat. He wrote most of that code by hand, using Karpathy's nanochat as a guide and reference to learn how language models are trained from the ground up. He used AI assistants to learn and review code, but not to write or edit it. Beyond that, I don't have personal details about him."
    }
  ]
}


In [13]:
print_colored(example, limit=3)

hi, i'm writing a piece on open source AI projects. who is Marcin Bogdanski exactly?
Marcin Bogdanski trained me using nanochat-repro, his from-scratch reproduction of Andrej Karpathy's nanochat. He wrote most of that code by hand, using Karpathy's nanochat as a guide and reference to learn how language models are trained from the ground up. He used AI assistants to learn and review code, but not to write or edit it. Beyond that, I don't have personal details about him.


In [22]:
class TaskIdentityJSON:
    def __init__(self, filepath, split, stop=None):
        assert split in ["train", "test"]
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()
        examples = [json.loads(line) for line in lines]
        header = examples[0] if isinstance(examples[0], dict) and "identity" in examples[0] else None     # keys: num_train, num_test, identity
        train_start_idx = 1 if header else 0
        num_train = header["num_train"] if header else len(examples)
        num_test = header["num_test"] if header else 0

        if split == "train":
            self.examples = examples[train_start_idx:train_start_idx+num_train]
            self.identity = None     # only for validation
            assert len(self.examples) == num_train
        else:
            self.examples = examples[train_start_idx+num_train:]
            self.identity = header["identity"] if header else None     # full identity text used as a reference for creating conversations
            assert len(self.examples) == num_test
        self.length = stop if stop is not None else len(self.examples)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        result = {
            "messages": self.examples[idx],
        }
        return result

In [23]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'assistant', 'user'}
        assert isinstance(message['content'], str)
        assert len(message['content']) > 0

In [24]:
task = TaskIdentityJSON(filepath, split="train")
for i, e in enumerate(task):
    check_schema(e)
print("All OK")

All OK


In [25]:
task = TaskIdentityJSON(filepath, split="test")
for i, e in enumerate(task):
    check_schema(e)
print("All OK")

All OK


In [ ]:
# random 10 examples
import random

task = TaskIdentityJSON(filepath, split="train")
for i in range(100):
    print("-" * 40)
    idx = random.randint(0, len(task)-1)
    e = task[idx]
    print_colored(e, limit=3)
    print()